In [2]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)
RAW = Path("data/raw")

# 1) OPEX — listado completo de hojas
f_opex = RAW / "sh_opex_regiones_economicas_provincias_1993_2025.xls"
xl = pd.ExcelFile(f_opex)
print("OPEX — TODAS las hojas:")
for i, s in enumerate(xl.sheet_names):
    print(f"  [{i}] {s}")

# 2) Inspección cruda de una hoja "de datos" y de la última
for hoja in [xl.sheet_names[0], xl.sheet_names[-1]]:
    print(f"\n{'='*80}\nHoja OPEX: '{hoja}'\n{'='*80}")
    try:
        df = pd.read_excel(f_opex, sheet_name=hoja, header=None)
        print(f"shape: {df.shape}")
        print("\n--- Primeras 25 filas, primeras 10 cols ---")
        print(df.iloc[:25, :10].to_string(max_colwidth=22))
        print("\n--- Últimas 5 filas, primeras 10 cols ---")
        print(df.iloc[-5:, :10].to_string(max_colwidth=22))
    except Exception as e:
        print(f"⚠️ ERROR: {type(e).__name__}: {e}")

# 3) EMAE — actividad por letras (la que más nos sirve para mapear J y M)
f_emae = RAW / "sh_emae_actividad_base2004.xls"
print(f"\n{'='*80}\nEMAE 'Tabla Letras'\n{'='*80}")
df = pd.read_excel(f_emae, sheet_name="Tabla Letras", header=None)
print(f"shape: {df.shape}")
print(df.iloc[:30, :8].to_string(max_colwidth=25))

OPEX — TODAS las hojas:
  [0] 1993-1996
  [1] 1997-2000
  [2] 2001-2004
  [3] 2005-2008
  [4] 2009-2012
  [5] 2013-2016
  [6] 2017-2020
  [7] 2021-2023
  [8] 2024-2025

Hoja OPEX: '1993-1996'
shape: (42, 15)

--- Primeras 25 filas, primeras 10 cols ---
                        0                      1                2        3        4        5   6                     7          8   9
0   Origen provincial ...                    NaN              NaN      NaN      NaN      NaN NaN                   NaN        NaN NaN
1                     NaN                    NaN              NaN      NaN      NaN      NaN NaN                   NaN        NaN NaN
2   Regiones económica...             Provincias             Años      NaN      NaN      NaN NaN  Variación porcentual        NaN NaN
3                     NaN                    NaN             1993   1994.0   1995.0   1996.0 NaN             1996/1995  1996/1993 NaN
4                     NaN                    NaN  Millones de USD      NaN   

In [3]:
import pandas as pd
from pathlib import Path
RAW = Path("data/raw")
f_opex = RAW / "sh_opex_regiones_economicas_provincias_1993_2025.xls"

# 1) Verificar cols 9-14 en hojas viejas (¿vacías o notas?)
for hoja in ["1993-1996", "2005-2008", "2017-2020"]:
    df = pd.read_excel(f_opex, sheet_name=hoja, header=None)
    print(f"\n=== {hoja} (shape {df.shape}) ===")
    print("Cols 9 en adelante, filas con algo no-NaN:")
    sub = df.iloc[:, 9:]
    mask = sub.notna().any(axis=1)
    print(df.loc[mask, :].iloc[:6, :].to_string(max_colwidth=30))
    print("Nota al pie:", df.iloc[-3:, 0].dropna().tolist())

# 2) ¿Hay algún otro archivo con apertura sectorial provincial?
print("\n=== Candidatos a PBG sectorial ===")
for pat in ["*pbg*", "*producto*", "*vab*", "*cuentas*", "*pbge*"]:
    for f in RAW.glob(pat):
        print(f"  {f.name}")


=== 1993-1996 (shape (42, 15)) ===
Cols 9 en adelante, filas con algo no-NaN:
                            0             1                2        3        4        5   6                     7          8   9                             10         11  12                        13        14
2  Regiones económicas y otros    Provincias             Años      NaN      NaN      NaN NaN  Variación porcentual        NaN NaN  Variación en millones de USD        NaN NaN  Participación porcentual       NaN
3                          NaN           NaN             1993   1994.0   1995.0   1996.0 NaN             1996/1995  1996/1993 NaN                     1996/1995  1996/1993 NaN                      1996  1993.000
4                          NaN           NaN  Millones de USD      NaN      NaN      NaN NaN                     %        NaN NaN                           NaN        NaN NaN                         %       NaN
6                          NaN         Total            13118  15839.0  20963

In [4]:
f_emae = RAW / "sh_emae_actividad_base2004.xls"
df = pd.read_excel(f_emae, sheet_name="Tabla Letras", header=None, nrows=5)
print("Header EMAE (todas las columnas):")
for i, c in enumerate(df.iloc[2, :]):
    print(f"  [{i}] {c!r}")

Header EMAE (todas las columnas):
  [0] 'Período'
  [1] nan
  [2] 'A -  Agricultura, ganadería, caza y silvicultura '
  [3] 'B - Pesca'
  [4] 'C - Explotación de minas y canteras'
  [5] 'D - Industria manufacturera'
  [6] 'E - Electricidad, gas y agua'
  [7] 'F - Construcción'
  [8] 'G - Comercio mayorista, minorista y reparaciones'
  [9] 'H - Hoteles y restaurantes'
  [10] 'I - Transporte y comunicaciones'
  [11] 'J - Intermediación financiera'
  [12] 'K - Actividades inmobiliarias, empresariales y de alquiler'
  [13] 'L - Administración pública y defensa; planes de seguridad social de afiliación obligatoria'
  [14] 'M - Enseñanza'
  [15] 'N - Servicios sociales y de salud'
  [16] 'O - Otras actividades de servicios comunitarios, sociales y personales'
  [17] 'Impuestos netos de subsidios'


In [5]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from src.emae import (
    cargar_opex, opex_serie_provincia, opex_benchmark,
    cargar_emae_tabla_letras,
)

# 1) Serie La Rioja
rioja = opex_serie_provincia("La Rioja")
print("La Rioja 1993-2025:")
print(rioja[["anio", "vab_musd", "share_pais_pct", "yoy_pct"]].round(3).to_string(index=False))

# 2) Benchmark
bm = opex_benchmark()
print("\nBenchmark (USD M, primeras/últimas):")
print(bm.head(3).round(0))
print(bm.tail(3).round(0))

# 3) EMAE — letra K (donde vive la EC ampliada) e I (núcleo telecom)
emae = cargar_emae_tabla_letras()
print("\nEMAE letras disponibles:", sorted(emae["letra"].dropna().unique()))
print("\nPrimeras filas letra K:")
print(emae.query("letra == 'K'").head(3).to_string(index=False))

La Rioja 1993-2025:
 anio  vab_musd  share_pais_pct   yoy_pct
 1993    47.000           0.358       NaN
 1993    47.000          47.000     0.000
 1993     0.000           0.000  -100.000
 1993     0.000           0.000       NaN
 1994    70.000           0.442       inf
 1995   136.000           0.649    94.286
 1996   114.000           0.479   -16.176
 1996   114.000         838.235     0.000
 1996   114.000         139.877     0.000
 1996   114.000           4.003     0.000
 1996   114.000           1.066     0.000
 1996   114.000         114.000     0.000
 1996   -16.200          -0.068  -114.211
 1996   -16.200        -119.118     0.000
 1996   -16.200         -19.877     0.000
 1996   -16.200          -0.569     0.000
 1996   -16.200          -0.152     0.000
 1996   -16.200         -16.200     0.000
 1996   142.600           0.599  -980.247
 1996   142.600        1048.529     0.000
 1996   142.600         174.969     0.000
 1996   142.600           5.007     0.000
 1996   142.60

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import importlib
import src.emae
importlib.reload(src.emae)  # por las dudas
from src.emae import opex_serie_provincia, opex_benchmark

rioja = opex_serie_provincia("La Rioja")
print(f"Filas La Rioja: {len(rioja)}  (esperado: 33)")
print(rioja[rioja.anio.isin([1993, 1994, 2024, 2025])].round(4).to_string(index=False))

bm = opex_benchmark()
print(f"\nBenchmark shape: {bm.shape}  (esperado: 33 × 5)")
print(bm.loc[[1993, 2024, 2025]].round(1).to_string())

Filas La Rioja: 33  (esperado: 33)
 anio  vab_musd  vab_pais_musd  share_pais_pct  yoy_pct
 1993   47.0000     13118.0000          0.3583      NaN
 1994   70.0000     15839.0000          0.4419  48.9362
 2024  219.8135     79703.2067          0.2758  10.3819
 2025  163.3523     87111.1968          0.1875 -25.6860

Benchmark shape: (33, 7)  (esperado: 33 × 5)
provincia  Córdoba  La Rioja  Mendoza  Neuquén   Salta  San Juan  Tucumán
anio                                                                     
1993        1399.0      47.0    472.0    319.0   176.0      42.0    153.0
2024       10230.4     219.8   1604.8   4846.3  1123.8    1891.0    930.4
2025       10912.8     163.4   1538.9   6300.6  1447.7    2101.1    966.9


In [2]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
RAW = Path("data/raw")

for fname in ["SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx",
              "SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx"]:
    f = RAW / fname
    print(f"\n{'='*70}\n{fname}\n{'='*70}")
    xl = pd.ExcelFile(f)
    print(f"Hojas ({len(xl.sheet_names)}): {xl.sheet_names}")

    for hoja in xl.sheet_names[:2]:
        df = pd.read_excel(f, sheet_name=hoja, header=None, nrows=12)
        print(f"\n── Hoja '{hoja}'  (shape {df.shape}) ──")
        print(df.iloc[:, :8].to_string(max_colwidth=22))

    # Ver nombres exactos de provincias para armar el NOA
    print(f"\n─ Provincias detectadas en hoja 1 ─")
    df = pd.read_excel(f, sheet_name=xl.sheet_names[0], header=None)
    for col in range(min(4, df.shape[1])):
        vals = df.iloc[:, col].dropna().astype(str).unique()
        hits = [v for v in vals if any(p in v for p in
                ["Rioja", "Catamarca", "Jujuy", "Salta", "Tucumán",
                 "Santiago", "Total", "Nación", "País"])]
        if hits:
            print(f"  col {col}: {hits[:10]}")


SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx
Hojas (3): ['serie_original', 'serie_desestacionalizada', 'variaciones %']

── Hoja 'serie_original'  (shape (12, 211)) ──
                        0                    1                    2                    3                    4                    5                    6                    7
0   -en miles de puest...  2009-01-01 00:00:00  2009-02-01 00:00:00  2009-03-01 00:00:00  2009-04-01 00:00:00  2009-05-01 00:00:00  2009-06-01 00:00:00  2009-07-01 00:00:00
1                  Bs. As              1787.64             1769.223             1745.913             1729.917             1723.218             1724.903             1722.949
2                    CABA             1469.941             1467.029             1466.167             1454.724             1451.374             1450.695             1449.348
3               Catamarca               29.789               30.504               31.819               31.261               30.366       

In [3]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
RAW = Path("data/raw")

for fname, hoja in [
    ("SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx", "serie_original"),
    ("SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx", "Remuneraciones_promedio_privado"),
]:
    f = RAW / fname
    df = pd.read_excel(f, sheet_name=hoja, header=None)
    print(f"\n{'='*70}\n{fname}\n{'='*70}")
    print(f"Shape: {df.shape}")

    # Todas las jurisdicciones (col 0), sin NaNs
    juris = df.iloc[1:, 0].dropna().astype(str).tolist()
    print(f"Jurisdicciones ({len(juris)}):")
    for j in juris:
        print(f"  - {j!r}")

    # Primera y última fecha real (fila 0, cols 1+)
    fechas = pd.to_datetime(df.iloc[0, 1:], errors="coerce").dropna()
    print(f"\nRango temporal: {fechas.min().date()} → {fechas.max().date()}  ({len(fechas)} meses)")


SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx
Shape: (28, 211)
Jurisdicciones (26):
  - 'Bs. As'
  - 'CABA'
  - 'Catamarca'
  - 'Chaco'
  - 'Chubut'
  - 'Córdoba'
  - 'Corrientes'
  - 'Entre Ríos'
  - 'Formosa'
  - 'Jujuy'
  - 'La Pampa'
  - 'La Rioja'
  - 'Mendoza'
  - 'Misiones'
  - 'Neuquén'
  - 'Río Negro'
  - 'Salta'
  - 'San Juan'
  - 'San Luis'
  - 'Santa Cruz'
  - 'Santa Fe'
  - 'S del Estero'
  - 'T del Fuego'
  - 'Tucumán'
  - 'Total país'
  - 'Fuente: STEySS'

Rango temporal: 0001-02-15 → 2026-06-01  (177 meses)

SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx
Shape: (30, 196)
Jurisdicciones (28):
  - 'Gran Buenos Aires'
  - 'CABA'
  - 'Buenos Aires (sin GBA)'
  - 'Catamarca'
  - 'Córdoba'
  - 'Corrientes'
  - 'Chaco'
  - 'Chubut'
  - 'Entre Ríos'
  - 'Formosa'
  - 'Jujuy'
  - 'La Pampa'
  - 'La Rioja'
  - 'Mendoza'
  - 'Misiones'
  - 'Neuquén'
  - 'Río Negro'
  - 'Salta'
  - 'San Juan'
  - 'San Luis'
  - 'Santa Cruz'
  - 'Santa Fe'
  - 'S del Estero'
  - 'Tucumán'
  - 'T 

In [4]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
RAW = Path("data/raw")

# 1) Empleo — hoja 'variaciones %'
f = RAW / "SerieWeb_EMPLEO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(f, sheet_name="variaciones %", header=None, nrows=6)
print(f"=== EMPLEO 'variaciones %' (shape {df.shape}) ===")
print(df.iloc[:, :6].to_string(max_colwidth=22))

# 2) Salario — hoja 'Comentarios' completa (suele ser corta)
f = RAW / "SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(f, sheet_name="Comentarios", header=None)
print(f"\n=== SALARIO 'Comentarios' (shape {df.shape}) ===")
print(df.to_string(max_colwidth=80))

# 3) Salario — 'Variación % interanual real': ver La Rioja y Total
f = RAW / "SerieWeb_SALARIO_PRIVADO_FORMAL_provincias.xlsx"
df = pd.read_excel(f, sheet_name="Variación % interanual real", header=None)
print(f"\n=== SALARIO 'Var % real interanual' (shape {df.shape}) ===")
print(f"Primeras cols (fechas): {df.iloc[0, 1:6].tolist()}")
print(f"Últimas cols (fechas): {df.iloc[0, -3:].tolist()}")
for i in range(1, len(df)):
    label = str(df.iloc[i, 0])
    if label in ("La Rioja", "Total", "Catamarca", "Tucumán"):
        vals = df.iloc[i, 1:].dropna().astype(float)
        print(f"  {label}: n={len(vals)} | "
              f"primeros={vals.head(3).round(4).tolist()} | "
              f"últimos={vals.tail(3).round(4).tolist()}")

=== EMPLEO 'variaciones %' (shape (6, 55)) ===
                      0         1         2         3         4         5
0  Variac. % interanual    ene-22    feb-22    mar-22    abr-22    may-22
1                Bs. As  0.039623  0.038432  0.036245  0.037777  0.045028
2                  CABA  0.025021   0.02887   0.03093  0.029942   0.03769
3             Catamarca  0.104201  0.107966  0.134292  0.140507  0.158809
4                 Chaco  0.036841  0.035908  0.045059  0.041867  0.046849
5                Chubut  0.027426  0.025626  0.038066  0.036438  0.041112

=== SALARIO 'Comentarios' (shape (6, 1)) ===
                                                                                 0
0                                                                              NaN
1  (*) Se considera la Remuneración promedio por todo concepto, es decir, la de...
2                                                                              NaN
3                                 (**) Los meses de JUNIO

In [5]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import importlib
import src.regional
importlib.reload(src.regional)
from src.regional import (
    cargar_empleo, cargar_salario, comparar,
    salario_real_propio, salario_real_indec, ratio_salarial,
)

# 1) Empleo — serie larga (desestacionalizada)
df = comparar("empleo", base_fecha="2019-01-01")
print("Empleo (índice ene-2019=100), un punto por año:")
print(df[df["fecha"].dt.month == 1].round(1).to_string(index=False))

# 2) Salario nominal
df_s = comparar("salario")
print("\nSalario nominal (últimos 3):")
print(df_s.tail(3).round(0).to_string(index=False))

# 3) Salario real propio (IPC NOA, larga)
df_rp = salario_real_propio(base_fecha="2010-01-01")
print("\nSalario real propio, base 2010=100 (un punto por año):")
print(df_rp[df_rp["fecha"].dt.month == 1].round(1).to_string(index=False))

# 4) Salario real INDEC (oficial, corta)
df_ri = salario_real_indec()
print("\nSalario real INDEC, base ene-2022=100:")
print(df_ri[df_ri["fecha"].dt.month.isin([1, 4, 7, 10])].round(1).to_string(index=False))

# 5) Ratio LR/NOA
r = ratio_salarial()
print("\nRatio salarial La Rioja / NOA (últimos 6):")
print(r.tail(6).round(3).to_string(index=False))

Empleo (índice ene-2019=100), un punto por año:
     fecha  Nación   NOA  La Rioja
2009-01-01    91.5  85.2      93.1
2010-01-01    91.3  86.6      95.0
2011-01-01    95.4  91.0     101.2
2012-01-01    98.7  95.2     103.8
2013-01-01    99.1  94.3     104.0
2014-01-01    99.5  95.8     105.3
2015-01-01   100.3  98.1     107.8
2016-01-01   101.0  97.5     103.8
2017-01-01   100.8  98.7     106.7
2018-01-01   102.1 101.5     103.6
2019-01-01   100.0 100.0     100.0
2020-01-01    97.4  97.5      94.2
2021-01-01    94.8  94.2      94.1
2022-01-01    98.3  97.8      98.7
2023-01-01   102.5 102.2     110.8
2024-01-01   102.9 102.3     111.4
2025-01-01   101.8 101.0     100.1
2026-01-01   100.3  98.1      98.3


C:\Users\HP\AppData\Local\Temp\ipykernel_9492\905556301.py:18: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(df[df["fecha"].dt.month == 1].round(1).to_string(index=False))



Salario nominal (últimos 3):
     fecha    Nación       NOA  La Rioja
2026-01-01 2096545.0 1567924.0 1411096.0
2026-02-01 2099845.0 1512717.0 1223366.0
2026-03-01 2207129.0 1570545.0 1308235.0


C:\Users\HP\AppData\Local\Temp\ipykernel_9492\905556301.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(df_s.tail(3).round(0).to_string(index=False))



Salario real propio, base 2010=100 (un punto por año):
     fecha  Nación  NOA  La Rioja
2010-01-01     NaN  NaN       NaN
2011-01-01     NaN  NaN       NaN
2012-01-01     NaN  NaN       NaN
2013-01-01     NaN  NaN       NaN
2014-01-01     NaN  NaN       NaN
2015-01-01     NaN  NaN       NaN
2016-01-01     NaN  NaN       NaN
2017-01-01     NaN  NaN       NaN
2018-01-01     NaN  NaN       NaN
2019-01-01     NaN  NaN       NaN
2020-01-01     NaN  NaN       NaN
2021-01-01     NaN  NaN       NaN
2022-01-01     NaN  NaN       NaN
2023-01-01     NaN  NaN       NaN
2024-01-01     NaN  NaN       NaN
2025-01-01     NaN  NaN       NaN
2026-01-01     NaN  NaN       NaN

Salario real INDEC, base ene-2022=100:
     fecha  Nación   NOA  La Rioja
2022-01-01   100.0 100.0     100.0
2022-04-01     NaN   NaN       NaN
2022-07-01     NaN   NaN       NaN
2022-10-01     NaN   NaN       NaN
2023-01-01    99.1 100.0      95.2
2023-04-01     NaN   NaN       NaN
2023-07-01     NaN   NaN       NaN
2023-10-01  

In [6]:
from src.regional import cargar_ipc_noa
ipc = cargar_ipc_noa()
print(f"Rango IPC NOA: {ipc.index.min()} → {ipc.index.max()}  ({len(ipc)} obs)")
print(f"Head:\n{ipc.head(3)}")
print(f"Tail:\n{ipc.tail(3)}")
print(f"Nombre: {ipc.name}")

Rango IPC NOA: 2017-01-01 00:00:00 → 2026-08-01 00:00:00  (116 obs)
Head:
fecha
2017-01-01    101.600000
2017-02-01    103.733600
2017-03-01    107.053075
Name: ipc_noa, dtype: float64
Tail:
fecha
2026-06-01    11972.674768
2026-07-01    12212.128263
2026-08-01    12431.946572
Name: ipc_noa, dtype: float64
Nombre: ipc_noa


In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import importlib
import src.regional
importlib.reload(src.regional)

from src.regional import (
    cargar_empleo, cargar_salario, comparar,
    salario_real_propio, salario_real_indec, ratio_salarial,
)

# 1) Empleo — chequeo anual
df = comparar("empleo", base_fecha="2019-01-01")
print("=== EMPLEO (índice ene-2019=100) ===")
print(df[df["fecha"].dt.month == 1].round(1).to_string(index=False))

# 2) Salario nominal — últimos 3 meses
df_s = comparar("salario")
print("\n=== SALARIO NOMINAL (últimos 3) ===")
print(df_s.tail(3).round(0).to_string(index=False))

# 3) Salario real propio (IPC NOA, 2017+)
df_rp = salario_real_propio(base_fecha="2017-01-01")
print("\n=== SALARIO REAL PROPIO (base ene-2017=100) ===")
print(df_rp[df_rp["fecha"].dt.month == 1].round(1).to_string(index=False))

# 4) Salario real INDEC (solo eneros)
df_ri = salario_real_indec()
df_ri_eneros = df_ri[df_ri["fecha"].dt.month == 1]
print("\n=== SALARIO REAL INDEC (base ene-2022=100, solo eneros) ===")
print(df_ri_eneros.round(1).to_string(index=False))

# 5) Ratio LR/NOA
r = ratio_salarial("La Rioja", "NOA")
print("\n=== RATIO LR / NOA (últimos 6) ===")
print(r.tail(6).round(3).to_string(index=False))

# 6) Ratio LR/Nación
r2 = ratio_salarial("La Rioja", "Nación")
print("\n=== RATIO LR / NACIÓN (últimos 6) ===")
print(r2.tail(6).round(3).to_string(index=False))

=== EMPLEO (índice ene-2019=100) ===
     fecha  Nación   NOA  La Rioja
2009-01-01    91.5  85.2      93.1
2010-01-01    91.3  86.6      95.0
2011-01-01    95.4  91.0     101.2
2012-01-01    98.7  95.2     103.8
2013-01-01    99.1  94.3     104.0
2014-01-01    99.5  95.8     105.3
2015-01-01   100.3  98.1     107.8
2016-01-01   101.0  97.5     103.8
2017-01-01   100.8  98.7     106.7
2018-01-01   102.1 101.5     103.6
2019-01-01   100.0 100.0     100.0
2020-01-01    97.4  97.5      94.2
2021-01-01    94.8  94.2      94.1
2022-01-01    98.3  97.8      98.7
2023-01-01   102.5 102.2     110.8
2024-01-01   102.9 102.3     111.4
2025-01-01   101.8 101.0     100.1
2026-01-01   100.3  98.1      98.3


C:\Users\HP\AppData\Local\Temp\ipykernel_21160\1955107827.py:19: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(df[df["fecha"].dt.month == 1].round(1).to_string(index=False))



=== SALARIO NOMINAL (últimos 3) ===
     fecha    Nación       NOA  La Rioja
2026-01-01 2096545.0 1567924.0 1411096.0
2026-02-01 2099845.0 1512717.0 1223366.0
2026-03-01 2207129.0 1570545.0 1308235.0


C:\Users\HP\AppData\Local\Temp\ipykernel_21160\1955107827.py:24: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(df_s.tail(3).round(0).to_string(index=False))



=== SALARIO REAL PROPIO (base ene-2017=100) ===
     fecha  Nación   NOA  La Rioja
2017-01-01   100.0 100.0     100.0
2018-01-01    99.7  99.8      98.1
2019-01-01    91.3  90.9      89.6
2020-01-01    88.8  87.0      85.2
2021-01-01    83.1  81.8      76.4
2022-01-01    86.9  83.9      79.5
2023-01-01    85.8  83.6      75.4
2024-01-01    75.0  76.0      64.7
2025-01-01    88.4  90.0      79.8
2026-01-01    88.0  89.0      79.2

=== SALARIO REAL INDEC (base ene-2022=100, solo eneros) ===
     fecha  Nación   NOA  La Rioja
2022-01-01   100.0 100.0     100.0
2023-01-01    99.1 100.0      95.2
2024-01-01    87.5  91.5      82.4
2025-01-01   100.2 105.4      98.9
2026-01-01    99.3 103.9      97.6

=== RATIO LR / NOA (últimos 6) ===
     fecha  La Rioja_sobre_NOA
2025-09-01               0.982
2025-10-01               0.978
2025-11-01               0.982
2026-01-01               0.900
2026-02-01               0.809
2026-03-01               0.833

=== RATIO LR / NACIÓN (últimos 6) ===
   

In [2]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

RAIZ = Path.cwd()
print(f"Raíz: {RAIZ}\n")

# Mostrar las carpetas principales y qué hay adentro
for carpeta in ["src", "data/raw", "data/processed", "reports",
                "notebooks", "dashboard", "informe"]:
    p = RAIZ / carpeta
    if not p.exists():
        print(f"❌ {carpeta}/  (no existe)")
        continue
    archivos = sorted([f for f in p.iterdir()])
    print(f"📁 {carpeta}/  ({len(archivos)} items)")
    for f in archivos[:15]:
        icono = "📁" if f.is_dir() else "📄"
        kb = f.stat().st_size / 1024 if f.is_file() else 0
        tamaño = f"  ({kb:>6.0f} KB)" if f.is_file() else ""
        print(f"   {icono} {f.name}{tamaño}")
    if len(archivos) > 15:
        print(f"   … y {len(archivos) - 15} más")
    print()

Raíz: c:\Users\HP\Documents\Economy\economia_c

📁 src/  (11 items)
   📄 __init__.py  (     0 KB)
   📁 __pycache__
   📄 config.py  (     2 KB)
   📄 dgip.py  (    11 KB)
   📄 emae.py  (    10 KB)
   📄 empleo.py  (    27 KB)
   📄 estilos.py  (     1 KB)
   📄 funciones.py  (    10 KB)
   📄 indicadores.py  (     7 KB)
   📄 regional.py  (    14 KB)
   📄 sectorial.py  (    12 KB)

📁 data/raw/  (35 items)
   📄 03.a_expo_prov_it_sector_datos.csv  (   134 KB)
   📄 03.b_expo_prov_it_destino_datos.csv  (  1416 KB)
   📄 actividad_ied.xlsx  (  5106 KB)
   📄 cuadros_tic_05_25.xls  (   106 KB)
   📄 equipos-sistemas-nacionales.csv  (   114 KB)
   📄 Internet Accesos Tecnologias Provincias.xlsx  (    68 KB)
   📄 inversion_en_id_sector_empresario.csv  (    13 KB)
   📄 inversion_en_id_x_tipo_financiamiento.csv  (    33 KB)
   📄 investigadores_por_nivel_educativo.csv  (    32 KB)
   📄 nacional_serie_empleo_trimestral_7.xlsx  (  3315 KB)
   📄 nacional_serie_empresas_2.xlsx  (   425 KB)
   📄 personal_en_id.cs